# AI Resume Screening Assistant
LangChain + RAG + FAISS + Streamlit. Run the cells **top to bottom**.

## Step 1 - Install libraries (run once)

In [1]:
!python -m pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu pypdf python-dotenv pydantic streamlit pandas

## Step 2 - Load your API key
Open the `.env` file in Notepad, type `GOOGLE_API_KEY=your-gemini-key-here`, save it, then run this cell.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Key not found - put GOOGLE_API_KEY=your-gemini-key in the .env file and SAVE it"
print("API key loaded")

API key loaded


## Step 3 - Load resumes (PDFs in the `resumes` folder)

In [12]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_files = sorted(Path("resumes").glob("*.pdf"))
print("Resumes found:", [p.name for p in pdf_files])

docs = []
for p in pdf_files:
    pages = PyPDFLoader(str(p)).load()
    for pg in pages:
        pg.metadata["candidate"] = p.name   # tag every page with its candidate
    docs.extend(pages)
print("Total pages loaded:", len(docs))

Resumes found: ['Data_Scientist_Resume_Medium.pdf', 'Data_Scientist_Resume_Strong.pdf', 'Data_Scientist_Resume_Weak.pdf', 'JyotirmayeeMahanta-3.pdf']
Total pages loaded: 5


## Step 4 - Split text into chunks

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print("Total chunks:", len(chunks))

Total chunks: 19


## Step 5 - Embeddings + FAISS vector database

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("vector_store")   # saved into your vector_store folder
print("Vector store ready")

Vector store ready


## Step 6 - Structured output, prompt and LLM

In [15]:
from typing import List
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

class Evaluation(BaseModel):
    match_score: int = Field(description="Overall fit with the JD, 0-100")
    candidate_summary: str = Field(description="2-3 sentence summary")
    matching_skills: List[str] = Field(description="JD skills found in the resume")
    missing_skills: List[str] = Field(description="JD skills NOT found in the resume")
    strengths: List[str]
    weaknesses: List[str]
    recommendation: str = Field(description="One of: Strong Hire, Hire, Maybe, No Hire")
    justification: str = Field(description="Short reasoning grounded in the resume")

parser = PydanticOutputParser(pydantic_object=Evaluation)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert technical recruiter. Evaluate the candidate ONLY using the resume "
               "excerpts provided. If something is not in the excerpts, treat it as missing - never "
               "invent experience.\n{format_instructions}"),
    ("human", "JOB DESCRIPTION:\n{jd}\n\nRESUME EXCERPTS ({candidate}):\n{context}\n\n"
              "Evaluate this candidate against the job description."),
])

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
chain = prompt | llm | parser

## Step 7 - Evaluation function (retriever + chain)

In [16]:
def evaluate_candidate(candidate, jd):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 6, "fetch_k": 60, "filter": {"candidate": candidate}})
    context = "\n---\n".join(d.page_content for d in retriever.invoke(jd))
    return chain.invoke({"jd": jd, "candidate": candidate, "context": context,
                         "format_instructions": parser.get_format_instructions()})

def show(name, e):
    print("=" * 70)
    print(f"{name}  |  Score: {e.match_score}/100  |  {e.recommendation}")
    print("Summary:", e.candidate_summary)
    print("Matching skills:", ", ".join(e.matching_skills))
    print("Missing skills :", ", ".join(e.missing_skills))
    print("Strengths      :", "; ".join(e.strengths))
    print("Weaknesses     :", "; ".join(e.weaknesses))
    print("Justification  :", e.justification)

## Step 8 - Job description (also saved into `job_descriptions` folder)

In [17]:
jd = """Data Scientist
We are looking for a Data Scientist with 2+ years of experience.
Required: Python, SQL, Pandas, NumPy, Scikit-learn, machine learning, statistics,
data visualization (Matplotlib/Seaborn/Power BI), model deployment.
Nice to have: deep learning (TensorFlow/PyTorch), NLP, LangChain / LLMs, cloud (AWS/Azure).
"""
Path("job_descriptions").mkdir(exist_ok=True)
Path("job_descriptions/data_scientist.txt").write_text(jd)
print("JD saved")

JD saved


## Test Case 1 - Evaluate Resume A for a Data Scientist role

In [18]:
names = [p.name for p in pdf_files]
resultA = evaluate_candidate(names[0], jd)
show(names[0], resultA)

Data_Scientist_Resume_Medium.pdf  |  Score: 55/100  |  No Hire
Summary: Ananya Sharma is a Computer Science postgraduate with strong foundational skills in Python, SQL, machine learning, and deep learning. She has hands-on experience developing end-to-end ML projects, including data preprocessing, model training, hyperparameter tuning, and deployment using tools like Scikit-learn, TensorFlow, Pandas, and Streamlit.
Matching skills: Python, SQL, Pandas, NumPy, Scikit-learn, machine learning, statistics, Matplotlib, Seaborn, Power BI, model deployment, deep learning (TensorFlow), NLP
Missing skills : 2+ years of experience, deep learning (PyTorch), LangChain / LLMs, cloud (AWS/Azure)
Strengths      : Strong proficiency in core data science libraries and languages (Python, SQL, Pandas, NumPy, Scikit-learn).; Demonstrated experience with end-to-end machine learning project development, including data preprocessing, model training, and evaluation.; Proficiency in multiple data visualization

## Test Case 2 - Compare Resume A and Resume B

In [19]:
resultB = evaluate_candidate(names[1], jd)
show(names[1], resultB)
better = names[0] if resultA.match_score >= resultB.match_score else names[1]
print("\nHigher score:", better)

Data_Scientist_Resume_Strong.pdf  |  Score: 85/100  |  Hire
Summary: Ananya Sharma is a recent Computer Science postgraduate with strong hands-on experience in end-to-end machine learning and deep learning solutions, including NLP and model deployment. Proficient in Python, SQL, and a comprehensive suite of data science tools, she demonstrates practical application through impactful projects.
Matching skills: Python, SQL, Pandas, NumPy, Scikit-learn, machine learning, statistics, Matplotlib, Seaborn, Power BI, model deployment, deep learning (TensorFlow/Keras), NLP
Missing skills : 2+ years of professional experience, PyTorch, LangChain / LLMs, cloud (AWS/Azure)
Strengths      : Comprehensive proficiency in all required technical skills (Python, SQL, Pandas, NumPy, Scikit-learn, ML, statistics, data visualization, model deployment).; Strong expertise in Deep Learning (TensorFlow/Keras, CNN, LSTM, GRU) and NLP, demonstrated through projects like Sarcasm Detection.; Proven ability to bui

## Test Case 3 - Missing skills in Resume C

In [20]:
resultC = evaluate_candidate(names[2], jd)
print(names[2], "missing skills:", resultC.missing_skills)
show(names[2], resultC)

Data_Scientist_Resume_Weak.pdf missing skills: ['2+ years of experience', 'statistics', 'model deployment', 'deep learning (TensorFlow/PyTorch)', 'NLP', 'LangChain / LLMs', 'cloud (AWS/Azure)']
Data_Scientist_Resume_Weak.pdf  |  Score: 45/100  |  No Hire
Summary: Ananya Sharma is an entry-level Computer Science postgraduate with a strong foundation in Python, SQL, and core machine learning libraries like Pandas, NumPy, and Scikit-learn. They have demonstrated practical application of these skills through academic projects involving regression, classification, data preprocessing, and visualization.
Matching skills: Python, SQL, Pandas, NumPy, Scikit-learn, machine learning, data visualization (Matplotlib/Seaborn/Power BI)
Missing skills : 2+ years of experience, statistics, model deployment, deep learning (TensorFlow/PyTorch), NLP, LangChain / LLMs, cloud (AWS/Azure)
Strengths      : Strong foundational skills in Python, SQL, Pandas, NumPy, and Scikit-learn.; Demonstrated ability to app

## Test Case 4 - Recommend the best candidate among all resumes

In [21]:
all_results = {n: evaluate_candidate(n, jd) for n in names}
evals = "\n\n".join(f"{n}: {e.model_dump_json()}" for n, e in all_results.items())
compare = ChatPromptTemplate.from_template(
    "You are a hiring manager. Using ONLY the evaluations below, recommend the best candidate and "
    "justify it in one short paragraph, mentioning the runner-up.\n\nJOB DESCRIPTION:\n{jd}\n\nEVALUATIONS:\n{evals}")
print((compare | llm | StrOutputParser()).invoke({"jd": jd, "evals": evals}))

Based on the evaluations, the best candidate is **Data_Scientist_Resume_Strong.pdf (Ananya Sharma)**. Despite not explicitly meeting the 2+ years of professional experience requirement, this candidate demonstrates an exceptional technical skill set, matching all required skills including model deployment, and showcasing strong capabilities in deep learning and NLP through impressive end-to-end projects. The evaluation explicitly states that their project work indicates readiness for the role, making them a strong candidate for an interview. The runner-up is **Data_Scientist_Resume_Medium.pdf (Ananya Sharma)**, who also possesses an excellent technical foundation and matches most required skills, but the evaluation notes the experience gap as a more significant mismatch, leading to a "Maybe" recommendation.


## Step 9 - Create the Streamlit app (writes `app.py`)
After running this cell, open **Anaconda Prompt / CMD** in your project folder and run `streamlit run app.py`.

In [22]:
%%writefile app.py
"""AI Resume Screening Assistant - LangChain + RAG + Streamlit."""
import os
import tempfile
from typing import List

import pandas as pd
import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

from pathlib import Path


def load_env():
    """Read GOOGLE_API_KEY from the .env file that sits next to app.py."""
    f = Path(__file__).parent / ".env"
    if f.exists():
        for line in f.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip('"').strip("'")


load_env()


# ---------- Structured output schema ----------
class Evaluation(BaseModel):
    match_score: int = Field(description="Overall fit with the JD, 0-100")
    candidate_summary: str = Field(description="2-3 sentence summary of the candidate")
    matching_skills: List[str] = Field(description="JD skills found in the resume")
    missing_skills: List[str] = Field(description="JD skills NOT found in the resume")
    strengths: List[str]
    weaknesses: List[str]
    recommendation: str = Field(description="One of: Strong Hire, Hire, Maybe, No Hire")
    justification: str = Field(description="Short reasoning grounded in the resume")


parser = PydanticOutputParser(pydantic_object=Evaluation)

EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert technical recruiter. Evaluate the candidate ONLY using the "
     "resume excerpts provided. If something is not in the excerpts, treat it as "
     "missing - never invent experience.\n{format_instructions}"),
    ("human",
     "JOB DESCRIPTION:\n{jd}\n\nRESUME EXCERPTS ({candidate}):\n{context}\n\n"
     "Evaluate this candidate against the job description."),
])

COMPARE_PROMPT = ChatPromptTemplate.from_template(
    "You are a hiring manager. Using ONLY the evaluations below, recommend the best "
    "candidate for the job and justify it in one short paragraph, mentioning the "
    "runner-up.\n\nJOB DESCRIPTION:\n{jd}\n\nEVALUATIONS:\n{evals}"
)

QA_PROMPT = ChatPromptTemplate.from_template(
    "Answer the recruiter's question ONLY from the resume excerpts below. Mention "
    "the candidate name for each fact. If the answer is not in the excerpts, say "
    "\"Not found in the uploaded resumes.\"\n\nEXCERPTS:\n{context}\n\nQUESTION: {question}"
)


# ---------- RAG pipeline ----------
def build_vectorstore(files, api_key):
    """Load PDFs -> split -> embed -> FAISS. Each chunk is tagged with its candidate."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = []
    for f in files:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(f.getvalue())
            path = tmp.name
        pages = PyPDFLoader(path).load()
        os.unlink(path)
        for c in splitter.split_documents(pages):
            c.metadata["candidate"] = f.name
            chunks.append(c)
    if not chunks:
        raise ValueError("No text could be extracted (scanned PDFs need OCR).")
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=api_key)
    return FAISS.from_documents(chunks, embeddings)


def format_docs(docs):
    return "\n---\n".join(d.page_content for d in docs)


def evaluate_candidate(vs, llm, candidate, jd):
    retriever = vs.as_retriever(
        search_kwargs={"k": 6, "fetch_k": 60, "filter": {"candidate": candidate}}
    )
    context = format_docs(retriever.invoke(jd))
    chain = EVAL_PROMPT | llm | parser
    return chain.invoke({
        "jd": jd, "candidate": candidate, "context": context,
        "format_instructions": parser.get_format_instructions(),
    })


# ---------- UI ----------
st.set_page_config(page_title="AI Resume Screener", page_icon="📄", layout="wide")
st.title("📄 AI Resume Screening Assistant")

api_key = os.getenv("GOOGLE_API_KEY", "")

with st.sidebar:
    model = st.selectbox("LLM", ["gemini-2.5-flash", "gemini-2.5-flash-lite"])

files = st.file_uploader("Upload resumes (PDF)", type="pdf", accept_multiple_files=True)
jd = st.text_area("Job Description", height=200,
                  placeholder="Paste the job description here...")

if st.button("Evaluate", type="primary"):
    if not (api_key and files and jd.strip()):
        st.error("Make sure GOOGLE_API_KEY is in your .env file, and add at least one resume, and a job description.")
    else:
        try:
            llm = ChatGoogleGenerativeAI(model=model, temperature=0, google_api_key=api_key)
            with st.spinner("Indexing resumes..."):
                st.session_state.vs = build_vectorstore(files, api_key)
            results = {}
            bar = st.progress(0.0)
            for i, f in enumerate(files):
                with st.spinner(f"Evaluating {f.name}..."):
                    results[f.name] = evaluate_candidate(st.session_state.vs, llm, f.name, jd)
                bar.progress((i + 1) / len(files))
            st.session_state.results = results
            st.session_state.jd = jd
            st.session_state.llm_model = model
            st.session_state.pop("best", None)
        except Exception as e:
            st.error(f"Something went wrong: {e}")

results = st.session_state.get("results")
if results:
    st.header("Results")
    ranked = sorted(results.items(), key=lambda kv: kv[1].match_score, reverse=True)

    df = pd.DataFrame([
        {"Candidate": n, "Match Score": e.match_score, "Recommendation": e.recommendation}
        for n, e in ranked
    ])
    st.dataframe(df, use_container_width=True, hide_index=True)

    for name, e in ranked:
        with st.expander(f"{name} - {e.match_score}/100 - {e.recommendation}"):
            st.progress(min(max(e.match_score, 0), 100) / 100)
            st.markdown(f"**Summary:** {e.candidate_summary}")
            c1, c2 = st.columns(2)
            with c1:
                st.markdown("**✅ Matching skills**")
                st.write(", ".join(e.matching_skills) or "None")
                st.markdown("**💪 Strengths**")
                for s in e.strengths:
                    st.markdown(f"- {s}")
            with c2:
                st.markdown("**❌ Missing skills**")
                st.write(", ".join(e.missing_skills) or "None")
                st.markdown("**⚠️ Weaknesses**")
                for w in e.weaknesses:
                    st.markdown(f"- {w}")
            st.markdown(f"**Justification:** {e.justification}")

    if len(results) > 1 and st.button("Recommend best candidate"):
        llm = ChatGoogleGenerativeAI(model=st.session_state.llm_model, temperature=0, google_api_key=api_key)
        evals = "\n\n".join(f"{n}: {e.model_dump_json()}" for n, e in ranked)
        st.session_state.best = (COMPARE_PROMPT | llm | StrOutputParser()).invoke(
            {"jd": st.session_state.jd, "evals": evals})
    if st.session_state.get("best"):
        st.success(st.session_state.best)

if "vs" in st.session_state:
    st.header("Ask about the resumes")
    q = st.text_input("e.g. Who has experience with PyTorch? Compare A and B on Python.")
    if q and api_key:
        llm = ChatGoogleGenerativeAI(model=model, temperature=0, google_api_key=api_key)
        docs = st.session_state.vs.similarity_search(q, k=8)
        context = "\n---\n".join(f"[{d.metadata['candidate']}] {d.page_content}" for d in docs)
        st.write((QA_PROMPT | llm | StrOutputParser()).invoke(
            {"context": context, "question": q}))


Overwriting app.py


In [ ]:
!python -m streamlit run app.py